# Therme Vals. Spatial Graph Analysis


### Peter Zumthor | Thermal Journey Network

In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

c:\Users\ramyayoub\Desktop\IAAC\Master\Semester-3\Graph ML -- DOCUMENTS\Graph ML --Ramy\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Check the TopologicPy Version

In [2]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.20) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [3]:
renderer = "vscode"

## 4. Load the file as a cluster

In [4]:
# room_objects = Topology.ByOBJPath(
#     r"C:\Users\ramyayoub\Desktop\IAAC\Master\Semester-3\Graph ML -- DOCUMENTS\Graph ML --Ramy\Graph ML -- Assignment\Assignment-01\01-Assets\ThermesVlas.obj",
#     selfMerge=False
# )

# cluster = room_objects[0]
# print("Loaded:", Topology.TypeAsString(cluster))


## 4.1 importing the layer names from rhino

In [ ]:
room_objects_grouped = Topology.ByOBJPath(
    r"C:\Users\ramyayoub\Desktop\IAAC\Master\Semester-3\Graph ML -- DOCUMENTS\Graph ML --Ramy\Graph ML -- Assignment\Assignment-01\01-Assets\ThermesVlas.obj",
    selfMerge=False
)

color_map = {
    "Hot_Room":     "red",
    "Warm_Room":    "orange",
    "Cold_Room":    "blue",
    "Neutral_Room": "green",
    "Corridor":     "gray"
}

all_tagged_cells = []
for topo in room_objects_grouped:
    d     = Topology.Dictionary(topo)
    name  = Dictionary.ValueAtKey(d, "name") if d else "unknown"
    color = color_map.get(name, "gray")
    
    merged = Topology.SelfMerge(topo)      
    cells  = Topology.Cells(merged) or []
    
    for cell in cells:
        d2   = Dictionary.ByKeysValues(["name", "color"], [name, color])
        cell = Topology.SetDictionary(cell, d2)
        all_tagged_cells.append(cell)
    
    print(f"  '{name}' → {len(cells)} cells → {color}")

print(f"Total tagged: {len(all_tagged_cells)}")

  'Hot_Room' → 6 cells → red
  'Warm_Room' → 9 cells → orange
  'Cold_Room' → 10 cells → blue
  'Neutral_Room' → 11 cells → green
  'Corridor' → 11 cells → gray
Total tagged: 47


## 5. extract each cell

In [6]:
cc = CellComplex.ByCells(all_tagged_cells)
print("CellComplex:", Topology.TypeAsString(cc))
cells_count = Topology.Cells(cc)
print(f"Cells in CellComplex: {len(cells_count)}")

CellComplex: CellComplex
Cells in CellComplex: 47


## 6.Load windows as apertures and add it to the CellComplex

In [7]:
window_objects = Topology.ByOBJPath(
    r"C:\Users\ramyayoub\Desktop\IAAC\Master\Semester-3\Graph ML -- DOCUMENTS\Graph ML --Ramy\Graph ML -- Assignment\Assignment-01\01-Assets\ThermesVlas_Windows.obj",
    selfMerge=True
)

windows = Topology.Faces(window_objects[0])
print(f"Windows found: {len(windows)}")

Windows found: 19


## 7. Load doors as apertures

In [8]:
door_objects = Topology.ByOBJPath(r"C:\Users\ramyayoub\Desktop\IAAC\Master\Semester-3\Graph ML -- DOCUMENTS\Graph ML --Ramy\Graph ML -- Assignment\Assignment-01\01-Assets\ThermesVlas_Doors.obj", selfMerge=True)
doors = Topology.Faces(door_objects[0])
print(f"Doors found: {len(doors)}")

Doors found: 56


In [17]:
cc = Topology.AddApertures(cc, windows, exclusive=False, subTopologyType="face")
cc = Topology.AddApertures(cc, doors,   exclusive=True, subTopologyType="face")

## 7. Build graph

In [18]:
m = Graph.ByTopology(
    cc,
    direct=False,
    viaSharedApertures=True,
    toExteriorApertures=True
)
print("Graph vertices:", len(Graph.Vertices(m)))
print("Edges:", len(Graph.Edges(m)))

Graph vertices: 120
Edges: 127


In [19]:
areas = [Cell.SurfaceArea(cell) for cell in all_tagged_cells]
min_area = min(areas)
max_area = max(areas)

vertices = Graph.Vertices(m)
for v in vertices:
    vx, vy, vz = Vertex.X(v), Vertex.Y(v), Vertex.Z(v)
    best_color = "gray"
    best_size  = 8
    best_dist  = float("inf")

    for cell, area in zip(all_tagged_cells, areas):
        c    = Topology.Centroid(cell)
        dist = ((vx-Vertex.X(c))**2 + (vy-Vertex.Y(c))**2 + (vz-Vertex.Z(c))**2)**0.5
        if dist < best_dist:
            best_dist  = dist
            cd         = Topology.Dictionary(cell)
            best_color = Dictionary.ValueAtKey(cd, "color") if cd else "gray"
            if best_color == "gray":
                best_size = 8
            else:
                best_size = 12 + int(48 * (area - min_area) / (max_area - min_area + 1))

    d = Dictionary.ByKeysValues(["size", "color"], [best_size, best_color])
    v = Topology.SetDictionary(v, d)

for e in Graph.Edges(m):
    d = Dictionary.ByKeysValues(["width", "color"], [2, "black"])
    e = Topology.SetDictionary(e, d)

In [20]:
Topology.Show([cc, m, windows, doors],
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.15,
              backgroundColor="white",
              width=1000, height=1000,
              renderer=renderer)

In [13]:
vertices = Graph.Vertices(m)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [10, "red"])
    v = Topology.SetDictionary(v, d)

edges = Graph.Edges(m)
for e in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [4, "blue"])
    e = Topology.SetDictionary(e, d)

Topology.Show([cc, m, doors, windows],
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.3,
              backgroundColor="white",
              width=800,
              height=800,
              renderer=renderer)